# 🌸 Kira: Manga Upscale & Kindle Adaptation Pipeline

**Kira** é um pipeline automatizado para melhorar a resolução de Mangás utilizando **Real-ESRGAN** (otimizado para artes de anime/mangá via GPU no Google Colab) e adaptá-los perfeitamente para e-readers **Amazon Kindle** usando **KCC (Kindle Comic Converter)**.

---

### 📁 Passo 1: Montar o Google Drive
Execute a célula abaixo para conectar a sua conta do Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montado com sucesso!")

### ⚙️ Passo 2: Instalar Dependências e o Kira
Instala o PyTorch com suporte a GPU, Real-ESRGAN, KCC (Kindle Comic Converter) e ferramentas de descompactação.

In [ ]:
# 1. Instalar pacotes de sistema e dependências
!apt-get update -qq && apt-get install -y -qq p7zip-full unrar > /dev/null
!pip install -q torch torchvision realesrgan natsort tqdm Pillow opencv-python rich click PyYAML rarfile
!pip install -q git+https://github.com/ciromattia/kcc.git

# 2. Clonar ou carregar o Kira
import sys, os
drive_kira = '/content/drive/MyDrive/kira'

if not os.path.exists('/content/kira'):
    if os.path.exists(drive_kira):
        !cp -r /content/drive/MyDrive/kira /content/kira
    else:
        !git clone https://github.com/Wather17/kira.git /content/kira || true

if os.path.exists('/content/kira') and '/content/kira' not in sys.path:
    sys.path.insert(0, '/content/kira')

if os.path.exists('/content/kira'):
    !pip install -q --no-deps -e /content/kira

print("✅ Todas as dependências e o Kira foram instalados com sucesso!")

### 🚀 Passo 3: Executar o Pipeline (Formulário Interativo)
Preencha as configurações abaixo e execute a célula para iniciar o upscale e a conversão dos seus mangás armazenados no Google Drive.

In [ ]:
#@title ⚙️ Configurações do Pipeline do Kira { display-mode: "form" }

import sys, os
if os.path.exists('/content/kira') and '/content/kira' not in sys.path:
    sys.path.insert(0, '/content/kira')

#@markdown **Diretórios no Google Drive:**
Manga_Input_Folder = "MyDrive/Manga_Inputs" #@param {type:"string"}
Kindle_Output_Folder = "MyDrive/Kindle_Outputs" #@param {type:"string"}

#@markdown **Configurações do Real-ESRGAN (Upscale IA):**
RealESRGAN_Model = "RealESRGAN_x4plus_anime_6B" #@param ["RealESRGAN_x4plus_anime_6B", "realesr-animevideov3", "RealESRGAN_x4plus"]
GPU_Tile_Size = 400 #@param {type:"integer"}
Grayscale_EInk = False #@param {type:"boolean"}
Max_Dimension_Px = 2400 #@param {type:"integer"}

#@markdown **Configurações do Kindle (KCC):**
Kindle_Device = "KPW5" #@param ["KPW5", "KPW3", "KO", "KS", "K11", "KV", "OTHER"]
Output_Format = "EPUB" #@param ["EPUB", "MOBI", "AZW3", "CBZ", "KFX"]
Gamma_Correction = 1.0 #@param {type:"number"}
Keep_Upscaled_CBZ = True #@param {type:"boolean"}

from kira.pipeline import MangaPipeline
from kira.cli import _print_summary
from pathlib import Path

print("▶️ Iniciando processamento...")

# Inicializar Pipeline
pipeline = MangaPipeline(
    model_name=RealESRGAN_Model,
    tile=GPU_Tile_Size,
    grayscale=Grayscale_EInk,
    max_dimension=Max_Dimension_Px,
    kindle_profile=Kindle_Device,
    output_format=Output_Format,
    gamma=Gamma_Correction,
    keep_upscaled_cbz=Keep_Upscaled_CBZ
)

# Processar diretório ou arquivo
results = pipeline.process_directory(Manga_Input_Folder, Kindle_Output_Folder)

print("\n🎉 Processamento concluído com sucesso!")
_print_summary(results)